# Snapshot eval statistics tables — CSV + LaTeX export across eval settings

Companion to `sps_eval_catplot_cmp.ipynb`: it picks up the **same sources** (keep the `SOURCES`
block in sync) and exports **one statistics table per metric**, with one row per (algorithm,
variant) and **all eval settings side by side as columns** — the numbers behind that notebook's
stat glyphs. Which settings are tabled is set by `EVAL_CFGS` (default: all discovered).

`SEEDS` narrows every source to one seed (`42`), a list (`[42, 43]`, pooled into each cell) or
all (`None`). The seed matched against is the one each run recorded in its **own metadata**
(sidecar JSON `"seed"` for `eval-ray`/`eval-sb`, `args.json` `"resolved_seed"` for `eval-rbc`) —
never the directory name. Sources with no run on a wanted seed are skipped with a warning, so
narrowing to fan-out seeds drops sources that only ever ran the default seed. Keep `SEEDS` in
sync with the catplot notebook whenever both should report the same episodes.

The export cell writes, per metric, `<date>_<time>_out_<i>.{csv,tex}` next to this notebook —
the CSV and LaTeX body of one table share the same `i`:

- a **CSV** with a flat `<setting>_<stat>` header (full precision: count, mean, std, median,
  min, max per setting), and
- a **LaTeX table body** (`.tex`) — one `mean ± std` and one `median` column per setting,
  rounded to 2 decimals, `\midrule` between algorithm groups — meant to be `\input{}` into a
  hand-written booktabs `tabular` skeleton (a commented example sits at the top of the file), so
  rounding and ± assembly live in code while the LaTeX wrapper stays a thin shell.

The body **ends with `\bottomrule`**: do NOT add another one after the `\input{}`. LaTeX inserts
file-hook tokens at the end of an `\input{}`ed file which start a new table cell, so a following
`\midrule`/`\bottomrule` fails with *Misplaced `\noalign`*. Close the skeleton with
`\end{tabular}` directly.

Here the sources are the nine `gen_lin_bat_power_pv` snapshots (PPO / SAC / DreamerV3 × spec /
N-way gen. / gen. distribution) on the six held-out battery-power × PV settings `eval_0` …
`eval_5` — see the catplot notebook for the variant and setting tables. With all six settings
a `.tex` body carries 2 × 6 = 12 value columns; to keep a table inside the A4 text block,
export one half at a time via `EVAL_CFGS`, e.g. one PV rating per table
(`("eval_0", "eval_2", "eval_4")` = 6 kW, `("eval_1", "eval_3", "eval_5")` = 12 kW).
Sources whose eval has not finished yet, and the RBC strategies until a rule-based eval has run
on `RBC_SNAPSHOT`, are skipped with a warning.

**Main input:** `SOURCES`, `EVAL_CFGS`, `SEEDS`, `METRICS` and `TEX_VARIANT_LABELS` in the next
cell.

In [ ]:
# --- Main input ---------------------------------------------------------------------
# One source per (algorithm, training variant) — keep SOURCES in sync with
# sps_eval_catplot_cmp.ipynb (the catplot notebook draws ONE eval setting per figure;
# this notebook tables the eval settings side by side).
# Keys per entry:
#   path          snapshot dir (absolute, or relative to the repo root)
#   eval-kind     "eval-ray" | "eval-sb" | "eval-rbc" — which runs/eval_* dirs to use
#   algo          first row key (also used in the export filenames)
#   variant       training-variant label, the second row key
#   runs          (optional) restrict to these runs/<name> directories; default: all
#   rbc-strategy  (eval-rbc only) which rule-based strategy to use

# snapshots/20260928_001132_gen_lin_bat_power_pv_all_ppo
# snapshots/20260928_001142_gen_lin_bat_power_pv_sampled_ppo
# snapshots/20260928_001151_gen_lin_bat_power_pv_spec_ppo
# snapshots/20260928_001235_gen_lin_bat_power_pv_all_sac
# snapshots/20260928_001243_gen_lin_bat_power_pv_sampled_sac
# snapshots/20260928_001255_gen_lin_bat_power_pv_spec_sac
# snapshots/20260928_001328_gen_lin_bat_power_pv_sampled_dreamerv3
# snapshots/20260928_001354_gen_lin_bat_power_pv_spec_dreamerv3
# snapshots/20260928_001505_gen_lin_bat_power_pv_all_dreamerv3

SOURCES = [
    # --- PPO -------------------------------------------------------------------
    {
        "path": "snapshots/20260928_001151_gen_lin_bat_power_pv_spec_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "spec",
    },
    {
        "path": "snapshots/20260928_001132_gen_lin_bat_power_pv_all_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "N-way gen.",
    },
    {
        "path": "snapshots/20260928_001142_gen_lin_bat_power_pv_sampled_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "gen. distribution",
    },
    # --- SAC -------------------------------------------------------------------
    {
        "path": "snapshots/20260928_001255_gen_lin_bat_power_pv_spec_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "spec",
    },
    {
        "path": "snapshots/20260928_001235_gen_lin_bat_power_pv_all_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "N-way gen.",
    },
    {
        "path": "snapshots/20260928_001243_gen_lin_bat_power_pv_sampled_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "gen. distribution",
    },
    # --- DreamerV3 (all three trainings stopped before 28 000 episodes) ----------
    {
        "path": "snapshots/20260928_001354_gen_lin_bat_power_pv_spec_dreamerv3",
        "eval-kind": "eval-ray",
        "algo": "DreamerV3",
        "variant": "spec",
    },
    {
        "path": "snapshots/20260928_001505_gen_lin_bat_power_pv_all_dreamerv3",
        "eval-kind": "eval-ray",
        "algo": "DreamerV3",
        "variant": "N-way gen.",
    },
    {
        "path": "snapshots/20260928_001328_gen_lin_bat_power_pv_sampled_dreamerv3",
        "eval-kind": "eval-ray",
        "algo": "DreamerV3",
        "variant": "gen. distribution",
    },
]

# Rule-based baselines: every strategy of this snapshot's eval_rbc run becomes one
# strip in the "RBC" x-axis group (drop entries here to trim the group). The
# rule-based eval is policy-independent — any of the nine snapshots works, they share
# the eval infra configs, reward and eval data schedule. No eval_rbc run exists for
# them yet (2026-09-30): until `--kind eval-rbc` has run on RBC_SNAPSHOT, every RBC
# source is skipped with a warning and the RBC group stays empty.
RBC_SNAPSHOT = "snapshots/20260928_001132_gen_lin_bat_power_pv_all_ppo"

RBC_STRATEGIES = (
    "do_nothing",
    "deficit_discharge",
    # "pv_surplus_charge",
    "self_coverage",
    "price_median",
    "price_median_scaled_0.25",
    # "price_median_scaled_0.5",
    # "price_median_scaled_0.75",
    "price_median_autarky",
)
SOURCES += [
    {"path": RBC_SNAPSHOT, "eval-kind": "eval-rbc", "algo": "RBC",
     "variant": strategy, "rbc-strategy": strategy}
    for strategy in RBC_STRATEGIES
]

# Which eval settings become the table columns. None = every setting discovered:
# held-out `eval_<i>` configs, numerically sorted (battery_lin_power_pv, capacity 10 kWh:
# eval_0 = BES 3.7 kW / PV 6 kW, eval_1 = 3.7 / 12, eval_2 = 8.0 / 6, eval_3 = 8.0 / 12,
# eval_4 = 11.0 / 6, eval_5 = 11.0 / 12); a flat STA layout appears as "default".
# Or an explicit tuple, e.g. ("eval_0", "eval_2", "eval_4") for the PV 6 kW half.
EVAL_CFGS: tuple[str, ...] | None = None

# Eval seeds to keep, applied to every source after discovery:
#   SEEDS = None       keep every discovered seed (default)
#   SEEDS = 42         keep only the eval run seeded 42
#   SEEDS = [42, 43]   keep those two seeds, pooled into each table cell
# Matched against the seed every run recorded itself (sidecar JSON "seed" for
# eval-ray/eval-sb, args.json "resolved_seed" for eval-rbc), never the directory name.
# A source with no run on any wanted seed is skipped with a warning, so narrowing to the
# `_seed<N>` fan-out seeds drops sources that only ever ran the default seed. Keep in
# sync with the catplot notebook whenever both should report the same episodes.
SEEDS: int | list[int] | None = 42

# One exported table per metric: rows (algorithm, variant), columns the eval settings.
METRICS = ("total_reward", "cum_E_kWh", "cum_price_EUR")

# Optional display names for the LaTeX export's variant column, keyed by the raw
# variant label. Only affects the .tex body — the CSV and the tables displayed in
# this notebook keep the raw keys. Values are inserted as RAW LaTeX (no escaping —
# math like $\times$ is allowed, escape %, &, _ yourself); unlisted variants keep
# their raw label, LaTeX-escaped. Shorter names keep the table inside the A4 text
# block without shrinking the font further.
TEX_VARIANT_LABELS: dict[str, str] = {
    "do_nothing": "do nothing",
    "deficit_discharge": "deficit discharge",
    "self_coverage": "self coverage",
    "price_median": "price median",
    "price_median_scaled_0.25": "price median $\\times$0.25",
    "price_median_scaled_0.5": "price median $\\times$0.5",
    "price_median_scaled_0.75": "price median $\\times$0.75",
    "price_median_autarky": "price median autarky",
}

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

EVAL_KINDS = ("eval-ray", "eval-sb", "eval-rbc")

# A held-out eval-config result directory ends in `_eval_<i>` (e.g. battery_lin_cap_eval_1);
# training-config directories (e.g. battery_lin_cap_3) do not match and are skipped.
_EVAL_CFG_DIR_PATTERN = re.compile(r"(?:^|_)eval_(\d+)$")


# Walk upwards from the cwd until the directory containing snapshots/ is found, so the
# notebook runs both from thesis_eval/ and from the repo root.
def _find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("No 'snapshots/' directory found upwards of the cwd — run inside the repo.")


REPO_ROOT = _find_repo_root()


# Maps a runs/eval_* directory to its eval kind — mirrors the run-id scheme
# tools/snapshot/submit_snapshot.py builds from --kind.
def _run_dir_eval_kind(run_dir: Path) -> str:
    if run_dir.name.startswith("eval_rbc_"):
        return "eval-rbc"
    if run_dir.name.startswith("eval_sb_"):
        return "eval-sb"
    return "eval-ray"


# `eval_<i>` for a held-out eval-config result dir name, None for anything else.
def _cfg_dir_setting(dir_name: str) -> str | None:
    match = _EVAL_CFG_DIR_PATTERN.search(dir_name)
    return f"eval_{match.group(1)}" if match else None


# Ray/SB3 eval CSVs carry a sidecar JSON (same stem) with the run's trial seed + algorithm.
def _sidecar_seed_and_algorithm(csv_path: Path) -> tuple[int | None, str | None]:
    sidecar = csv_path.with_suffix(".json")
    if not sidecar.is_file():
        return None, None
    try:
        payload = json.loads(sidecar.read_text())
    except (json.JSONDecodeError, OSError):
        return None, None
    return payload.get("seed"), payload.get("algorithm")


# Rule-based runs record resolved_seed in an args.json above episodes.csv (directly
# above the per-strategy dir in the flat layout, one level further up with per-config
# dirs) — walk up towards the run dir and take the first hit.
def _rbc_resolved_seed(csv_path: Path, run_dir: Path) -> int | None:
    for candidate_dir in csv_path.parents:
        args_path = candidate_dir / "args.json"
        if args_path.is_file():
            try:
                return json.loads(args_path.read_text()).get("resolved_seed")
            except (json.JSONDecodeError, OSError):
                return None
        if candidate_dir == run_dir:
            return None
    return None


# Discovers one record per eval CSV of `source` across ALL eval settings kept by
# EVAL_CFGS (None = no filter). Both layouts are globbed; every record is tagged with
# its setting ("default" for the flat STA layout, `eval_<i>` for held-out GEN config
# dirs).
def _discover_source_records(source: dict) -> list[dict]:
    eval_kind = source["eval-kind"]
    if eval_kind not in EVAL_KINDS:
        raise ValueError(f"'eval-kind' must be one of {EVAL_KINDS}, got {eval_kind!r}")

    snapshot_dir = Path(source["path"])
    snapshot_dir = snapshot_dir if snapshot_dir.is_absolute() else REPO_ROOT / snapshot_dir
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory not found: {snapshot_dir}")

    run_dirs = [
        run_dir for run_dir in sorted((snapshot_dir / "runs").glob("eval_*"))
        if _run_dir_eval_kind(run_dir) == eval_kind
    ]
    if wanted_runs := source.get("runs"):
        run_dirs = [run_dir for run_dir in run_dirs if run_dir.name in set(wanted_runs)]
    if not run_dirs:
        raise FileNotFoundError(f"No '{eval_kind}' run directories found under {snapshot_dir / 'runs'}")

    records = []
    for run_dir in run_dirs:
        if eval_kind == "eval-rbc":
            # flat: .../<strategy>/episodes.csv — nested: .../<strategy>/<cfg>_eval_<i>/episodes.csv
            candidates = [
                (csv_path, "default", csv_path.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/episodes.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name), csv_path.parent.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/*/episodes.csv"))
            ]
            strategies = sorted({strategy for _, _, strategy in candidates})
            strategy_wanted = source.get("rbc-strategy")
            if strategy_wanted is None and len(strategies) > 1:
                raise ValueError(
                    f"Source '{source['algo']} — {source['variant']}': multiple rule-based strategies "
                    f"{strategies} under {run_dir.name} — set 'rbc-strategy' to pick one."
                )
            for csv_path, setting, strategy in candidates:
                if setting is None or (strategy_wanted is not None and strategy != strategy_wanted):
                    continue
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": _rbc_resolved_seed(csv_path, run_dir),
                    "eval_algorithm": strategy, "csv": csv_path,
                })
        else:
            candidates = [
                (csv_path, "default")
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/eval_*.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name))
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/*/eval_*.csv"))
            ]
            for csv_path, setting in candidates:
                if setting is None:
                    continue  # training-config result dir — not a held-out eval config
                seed, algorithm = _sidecar_seed_and_algorithm(csv_path)
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": seed,
                    "eval_algorithm": algorithm, "csv": csv_path,
                })

    available = sorted({record["setting"] for record in records})
    if EVAL_CFGS is not None:
        records = [record for record in records if record["setting"] in set(EVAL_CFGS)]
    if not records:
        raise FileNotFoundError(
            f"Source '{source['algo']} — {source['variant']}': no eval CSVs on setting(s) "
            f"{EVAL_CFGS} under {snapshot_dir} (available: {available or 'none'})."
        )
    return records


_source_keys = [(source["algo"], source["variant"]) for source in SOURCES]
if len(_source_keys) != len(set(_source_keys)):
    raise ValueError(f"Every source needs a unique (algo, variant) pair; got {_source_keys}")

# SEEDS normalised to the set of wanted seeds (None = keep every discovered seed): a
# bare int selects one seed, any iterable of ints selects several.
SEED_FILTER: set[int] | None = (
    None if SEEDS is None else {int(seed) for seed in ([SEEDS] if isinstance(SEEDS, int) else SEEDS)}
)
if SEED_FILTER is not None and not SEED_FILTER:
    raise ValueError("SEEDS is empty — use None to keep every discovered seed, or name the seed(s) to keep.")

# Pool a source's eval runs — narrowed to SEED_FILTER when set — but keep only the
# first CSV per (setting, seed): the same checkpoint evaluated with the same seed on
# the same setting replays the identical eval episodes, so a duplicate would
# double-count them. Seed-less CSVs are kept and flagged while no seed filter is active
# (with one they cannot match, so they are dropped). A source left with nothing — eval
# run still pending, or no run on any wanted seed — is skipped with a warning.
eval_records: list[dict] = []
print(f"Repo root    : {REPO_ROOT}")
print(f"Eval settings: {'all discovered' if EVAL_CFGS is None else EVAL_CFGS}")
print(f"Eval seeds   : {sorted(SEED_FILTER) if SEED_FILTER is not None else 'all discovered (SEEDS = None)'}")
for source in SOURCES:
    try:
        discovered = sorted(_discover_source_records(source),
                            key=lambda record: (record["setting"], record["run"], record["csv"].name))
    except FileNotFoundError as error:
        print(f"\nWARNING: source '{source['algo']} — {source['variant']}' skipped — {error}")
        continue

    if SEED_FILTER is not None:
        for record in discovered:
            if record["seed"] is None:
                print(f"note: source '{source['algo']} — {source['variant']}': dropped {record['run']} / "
                      f"{record['csv'].name} on {record['setting']} (no seed resolvable — cannot match SEEDS)")
        seeds_available = sorted({record["seed"] for record in discovered if record["seed"] is not None})
        discovered = [record for record in discovered if record["seed"] in SEED_FILTER]
        if not discovered:
            print(f"\nWARNING: source '{source['algo']} — {source['variant']}' skipped — no eval CSVs with a "
                  f"seed in {sorted(SEED_FILTER)} (seeds available: {seeds_available or 'none'}) — "
                  "adjust SEEDS or the source.")
            continue

    kept, seen = [], set()
    for record in discovered:
        if record["seed"] is not None and (record["setting"], record["seed"]) in seen:
            print(f"note: source '{source['algo']} — {source['variant']}': dropped {record['run']} / "
                  f"{record['csv'].name} on {record['setting']} (duplicate seed {record['seed']})")
            continue
        if record["seed"] is None:
            print(f"WARNING: source '{source['algo']} — {source['variant']}': no seed resolvable for "
                  f"{record['csv'].name} — kept, duplicate eval runs undetectable")
        else:
            seen.add((record["setting"], record["seed"]))
        kept.append(record)
    eval_records.extend(kept)

    strategy = f", strategy={source.get('rbc-strategy')}" if source["eval-kind"] == "eval-rbc" else ""
    print(f"\nSource '{source['algo']} — {source['variant']}' ({source['eval-kind']}{strategy}): {source['path']}")
    for record in kept:
        print(f"  {record['setting']:>8}  seed {str(record['seed']):>5}  <-  {record['run']} / {record['csv'].name}")

if not eval_records:
    raise FileNotFoundError("No eval CSVs discovered for any source — nothing to table.")

# Row order = first-appearance order in SOURCES, restricted to sources with results.
_algos_with_records = {record["algo"] for record in eval_records}
_variants_with_records = {record["variant"] for record in eval_records}
algo_labels = list(dict.fromkeys(source["algo"] for source in SOURCES if source["algo"] in _algos_with_records))
variant_labels = list(dict.fromkeys(source["variant"] for source in SOURCES if source["variant"] in _variants_with_records))


# Column order: the flat "default" setting first (if present), then eval_<i> numerically.
def _setting_sort_key(setting: str) -> tuple[int, int]:
    match = re.fullmatch(r"eval_(\d+)", setting)
    return (0, 0) if match is None else (1, int(match.group(1)))


setting_labels = sorted({record["setting"] for record in eval_records}, key=_setting_sort_key)
print(f"\nTable columns (eval settings): {setting_labels}")


## Output naming

Everything saved here lands next to the notebook as `<date>_<time>_out_<i>.<file format>`:
one timestamp per notebook run, `out_index` counting up once per exported table (its CSV and
`.tex` share that index), so no run overwrites an earlier one.

In [ ]:
from datetime import datetime
from pathlib import Path

# VS Code exposes the notebook path; a plain Jupyter kernel starts in the notebook's directory.
OUT_DIR = Path(globals()["__vsc_ipynb_file__"]).parent if "__vsc_ipynb_file__" in globals() else Path.cwd()
OUT_RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
out_index = 0  # incremented once per saved artefact; reset by re-running this cell


def next_out_stem() -> Path:
    """Next '<date>_<time>_out_<i>' path stem in this notebook's directory.

    One index per exported table — its CSV and LaTeX body differ only in the suffix."""
    global out_index
    out_index += 1
    return OUT_DIR / f"{OUT_RUN_STAMP}_out_{out_index}"


## Load per-episode metrics

The kept CSVs are concatenated into one frame with a row per eval episode and the metrics as
columns, each row tagged with its `algo`, training `variant`, `eval_cfg` (the setting) and eval
`seed`. The preview below counts the episodes behind each table cell.

In [ ]:
frames = []
for record in eval_records:
    csv_frame = pd.read_csv(record["csv"])
    present = [metric for metric in METRICS if metric in csv_frame.columns]
    missing = [metric for metric in METRICS if metric not in csv_frame.columns]
    if missing:
        print(f"note: source '{record['algo']} — {record['variant']}' {record['run']} lacks {missing} — "
              "skipped in those tables")
    frame = csv_frame[present].copy()
    frame["algo"] = record["algo"]
    frame["variant"] = record["variant"]
    frame["eval_cfg"] = record["setting"]
    frame["seed"] = record["seed"]
    frames.append(frame)

episodes = pd.concat(frames, ignore_index=True)
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

# Episodes behind each table cell: rows (algo, variant), columns the eval settings.
episodes.groupby(["algo", "variant", "eval_cfg"]).size().unstack("eval_cfg").reindex(
    columns=setting_labels
).reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


## Per-strip statistics across the eval settings — display and export

One table per metric: rows = (algorithm, variant), columns = (eval setting, stat). The cell
below displays each table rounded to 3 decimals and writes the CSV (full precision) plus the
LaTeX body (2 decimals) next to this notebook. The variant column of the `.tex` uses the display
names from `TEX_VARIANT_LABELS`; the CSV and the displayed tables keep the raw variant keys.

In [ ]:
from IPython.display import display

_STATS = ("count", "mean", "std", "median", "min", "max")


# Wide per-metric table: rows (algo, variant), columns (eval setting, stat).
def _metric_stats_wide(metric: str) -> pd.DataFrame:
    stats = episodes.groupby(["algo", "variant", "eval_cfg"])[metric].agg(list(_STATS))
    wide = stats.unstack("eval_cfg").swaplevel(axis=1)
    wide = wide.reindex(columns=pd.MultiIndex.from_product([setting_labels, _STATS]))
    return wide.reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


# LaTeX-escape the label columns (the RBC strategy names carry underscores).
def _latex_escape(text: str) -> str:
    return text.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")


# One table cell: "$mean \pm std$" at 2 decimals; bare mean when std is undefined
# (single-episode cell), "--" when a (row, setting) combination has no episodes.
def _latex_cell(mean: float, std: float | None = None) -> str:
    if pd.isna(mean):
        return "--"
    if std is not None and pd.notna(std):
        return f"${mean:.2f} \\pm {std:.2f}$"
    return f"${mean:.2f}$"


# Table body — rounding and ± assembly happen here, once, so the hand-written
# tabular skeleton that \input{}s this stays thin. \midrule separates the algorithm
# groups; the algorithm label is printed on its group's first row only. The file ends
# with \bottomrule ON PURPOSE: LaTeX inserts file-hook tokens at the end of an
# \input{}ed file, and after the last row's \\ those tokens start a new cell — a
# \bottomrule placed after \input{} in the skeleton therefore dies with
# "Misplaced \noalign". Inside the file it is safe; \end{tabular} may follow
# \input{} directly.
def _stats_latex_body(metric: str, wide: pd.DataFrame) -> str:
    lines = [
        f"% Auto-generated by snapshot_eval_stats_table_algo_compare.ipynb — booktabs body for '{metric}'",
        "% incl. trailing \\bottomrule (a \\bottomrule AFTER \\input{} would be 'Misplaced \\noalign').",
        "% Columns: algorithm & variant"
        + "".join(f" & {setting} mean+-std & {setting} median" for setting in setting_labels) + ".",
        "% Full precision, count, min and max: see the CSV with the same stem.",
        "% Verified skeleton:",
        f"%   \\begin{{tabular}}{{ll*{{{2 * len(setting_labels)}}}{{c}}}}",
        "%     \\toprule",
        "%     Algorithm & Variant & ... column heads ... \\\\",
        "%     \\midrule",
        "%     \\input{<this file>}",
        "%   \\end{tabular}",
    ]
    previous_algo = None
    for (algo, variant), row in wide.iterrows():
        if previous_algo is not None and algo != previous_algo:
            lines.append(r"\midrule")
        # TEX_VARIANT_LABELS values are raw LaTeX (no escaping); unmapped variants
        # fall back to their raw label, LaTeX-escaped.
        variant_cell = TEX_VARIANT_LABELS.get(variant, _latex_escape(str(variant)))
        cells = [_latex_escape(str(algo)) if algo != previous_algo else "", variant_cell]
        for setting in setting_labels:
            cells.append(_latex_cell(row[(setting, "mean")], row[(setting, "std")]))
            cells.append(_latex_cell(row[(setting, "median")]))
        lines.append(" & ".join(cells) + r" \\")
        previous_algo = algo
    lines.append(r"\bottomrule")
    return "\n".join(lines) + "\n"


for metric in present_metrics:
    wide = _metric_stats_wide(metric)
    export_frame = wide.copy()
    export_frame.columns = [f"{setting}_{stat}" for setting, stat in wide.columns]
    latex_body = _stats_latex_body(metric, wide)
    out_stem = next_out_stem()  # both formats of one table share the index
    export_frame.to_csv(f"{out_stem}.csv")
    Path(f"{out_stem}.tex").write_text(latex_body)
    print(f"saved {out_stem}.{{csv,tex}}  ({metric})")
    print(f"\n=== {metric} ===")
    display(wide.round(3))
